<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/3_1_river_boca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

En esta parte junto las planillas de los clubes Boca Juniors y River Plate para que aparezcan como una sola y hacer la investigacion mas sencilla

In [ ]:
"""
Une los CSV anuales (2021-2025) de Boca y River.

Genera en la carpeta de salida:
  boca_2021_2025.csv, river_2021_2025.csv y river_boca_2021_2025.csv

Busca los CSV en la carpeta indicada Y en todas sus subcarpetas, y deja
las planillas unidas en esa misma carpeta.

Uso (desde la carpeta donde están los CSV):
    python unir_csv.py
O indicando la carpeta (con comillas si tiene espacios):
    python unir_csv.py "datos/Temporadas River - Boca - Seleccion"

Requiere: pip install pandas
"""
import re
import sys
from pathlib import Path

import pandas as pd

EQUIPOS = {"boca": "Boca Juniors", "river": "River Plate"}
PATRON_GOLES = r"^\s*(\d+)\s*(?:\((\d+)\))?\s*$"  # "1" o "1 (4)" (con penales)


def leer_csv(archivo):
    """Lee un CSV probando utf-8 y, si falla, latin-1."""
    for codificacion in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(archivo, encoding=codificacion)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No pude leer {archivo.name}")


def main():
    # Une todos los argumentos: así funciona aunque la ruta tenga espacios sin comillas
    entrada = Path(" ".join(sys.argv[1:]).strip() or ".").expanduser()
    salida = entrada

    if not entrada.is_dir():
        sys.exit(
            f"Error: la carpeta no existe: {entrada.resolve()}\n"
            f"Carpeta actual: {Path.cwd()}\n"
            "Revisá la ruta (con comillas si tiene espacios) o ejecutá el script "
            "desde la carpeta donde están los CSV."
        )

    por_equipo = []
    for clave, nombre in EQUIPOS.items():
        partes = []
        columnas_ref = None
        for archivo in sorted(entrada.rglob(f"{clave}_*.csv")):
            coincide = re.fullmatch(rf"{clave}_(\d{{4}})", archivo.stem)
            if not coincide:  # ignora archivos ya unidos (ej: boca_2021_2025)
                continue
            anio = coincide.group(1)
            df = leer_csv(archivo)

            # verifica que todos los años tengan las mismas columnas
            if columnas_ref is None:
                columnas_ref = list(df.columns)
            elif list(df.columns) != columnas_ref:
                sys.exit(f"Error: {archivo.name} tiene columnas distintas al resto.")

            df.insert(0, "Equipo", nombre)
            df.insert(1, "Temporada", int(anio))
            partes.append(df)
            print(f"{archivo.relative_to(entrada)}: {len(df)} filas")

        if not partes:
            print(f"Aviso: no encontré archivos de {nombre} en {entrada}\n")
            continue

        unido = pd.concat(partes, ignore_index=True)
        unido["Date"] = pd.to_datetime(unido["Date"])

        # En partidos definidos por penales GF/GA vienen como "1 (4)":
        # se separan en goles (GF/GA) y penales (Pen_GF/Pen_GA).
        for col in ("GF", "GA"):
            partes_gol = unido[col].astype(str).str.extract(PATRON_GOLES)
            unido[col] = pd.to_numeric(partes_gol[0]).astype("Int64")
            unido[f"Pen_{col}"] = pd.to_numeric(partes_gol[1]).astype("Int64")
        for col in ("Poss", "Attendance"):  # Int64 evita 67.0 en vez de 67
            unido[col] = pd.to_numeric(unido[col], errors="coerce").astype("Int64")

        unido = unido.sort_values("Date", kind="stable").reset_index(drop=True)

        # utf-8-sig para que Excel muestre bien las tildes
        unido.to_csv(salida / f"{clave}_2021_2025.csv", index=False, encoding="utf-8-sig")
        print(f"-> {clave}_2021_2025.csv: {len(unido)} filas\n")
        por_equipo.append(unido)

    if not por_equipo:
        sys.exit(
            "Error: no encontré archivos tipo boca_2021.csv / river_2021.csv "
            f"en {entrada.resolve()}"
        )

    if len(por_equipo) > 1:
        todo = pd.concat(por_equipo, ignore_index=True)
        todo = todo.sort_values(["Date", "Equipo"], kind="stable").reset_index(drop=True)
        todo.to_csv(salida / "river_boca_2021_2025.csv", index=False, encoding="utf-8-sig")
        print(f"-> river_boca_2021_2025.csv: {len(todo)} filas")


if __name__ == "__main__":
    main()